In [ ]:
import sys, subprocess
print(sys.version)
subprocess.run(['nvidia-smi'])


In [ ]:
import subprocess, pathlib
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version,compute_cap','--format=csv,noheader'], text=True))
repo = pathlib.Path('/content/IsaacLab')
if not repo.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/isaac-sim/IsaacLab.git',str(repo)], check=True)
subprocess.run(['git','fetch','--depth','1','origin','5ffd3d5ac0b4d77d31ea820930a8f5fcac673d87'],cwd=repo,check=True)
subprocess.run(['git','checkout','--detach','5ffd3d5ac0b4d77d31ea820930a8f5fcac673d87'],cwd=repo,check=True)
print('Isaac Lab commit:', subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True))
print((repo/'AGENTS.md').read_text())


In [ ]:
# Official Newton-only setup; keeps Colab's Python environment separate.
import sys, subprocess, pathlib
subprocess.run([sys.executable,'-m','pip','install','-q','uv'], check=True)
setup_log = pathlib.Path('/content/isaaclab-setup.log')
with setup_log.open('w') as log:
    result = subprocess.run(['uv','sync'], cwd='/content/IsaacLab', stdout=log, stderr=subprocess.STDOUT)
print(setup_log.read_text()[-6000:])
result.check_returncode()
print('SETUP COMPLETE')


## Prediction recorded before training — 20 October 2026

Widening both linear command axes from [-1, 1] to [-2, 2] m/s at the same training budget will likely increase error_vel_xy and decrease success_rate, because the robot must learn to track harder commands and receives fewer samples from the original easier range. Base-contact terminations may increase if higher requested speeds destabilize the gait. This last prediction is less certain.

Three times as many iterations should improve performance if the main limitation is insufficient learning experience or optimization. It may not fully solve a limit imposed by the robot dynamics, action limits, or existing reward tradeoffs. Improvement would support, but not prove, an optimization/data explanation; no improvement would not by itself rule out every benefit from more data.

Plan: baseline with default 4096 environments and 300 iterations; wide range with 300 iterations; wide range from a fresh initialization with 900 iterations. Preserve all other settings. Log the exact revision and resolved configurations. Training has not begun.


In [ ]:
# Check the isolated Python/CUDA environment and the installed training interface.
import subprocess
for command in [
    ['uv','run','python','-c',"import sys,torch,warp; print(sys.version); print('PyTorch:',torch.__version__); print('CUDA:',torch.cuda.is_available()); print('GPU:',torch.cuda.get_device_name(0)); warp.init()"],
    ['uv','run','isaaclab','train','--rl_library','rsl_rl','--help'],
]:
    result = subprocess.run(command,cwd='/content/IsaacLab',capture_output=True,text=True)
    print(result.stdout[-16000:]); print(result.stderr[-4000:]); result.check_returncode()


In [ ]:
# Run A: all task and PPO defaults; only the logging name is customized.
import subprocess, pathlib, os, json, datetime
repo = pathlib.Path('/content/IsaacLab')
artifacts = pathlib.Path('/content/go2-senior'); artifacts.mkdir(exist_ok=True)
manifest = {'started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(), 'isaaclab_commit':subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip(), 'gpu':subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version,compute_cap','--format=csv,noheader'],text=True).strip(), 'task':'Isaac-Velocity-Flat-UnitreeGo2'}
(artifacts/'environment.json').write_text(json.dumps(manifest,indent=2))
baseline_command = ['uv','run','--no-sync','isaaclab','train','--rl_library','rsl_rl','--task',manifest['task'],'--run_name','baseline','--logger','tensorboard']
train_log = artifacts/'baseline-console.log'
with train_log.open('w') as log:
    train_job = subprocess.Popen(baseline_command,cwd=repo,stdout=log,stderr=subprocess.STDOUT,env={**os.environ,'PYTHONUNBUFFERED':'1'})
(artifacts/'baseline-command.json').write_text(json.dumps(baseline_command,indent=2))
print('Baseline training started. PID:',train_job.pid)
print('Defaults: 4096 environments; 300 iterations. No pretrained checkpoint loaded.')


In [ ]:
# Re-run this cell to inspect training.
import psutil
print('Training process exit code (None = running):', train_job.poll())
print(train_log.read_text(errors='replace')[-2500:])
p = psutil.Process(train_job.pid) if train_job.poll() is None else None
if p:
    print('Process elapsed seconds:', round(__import__('time').time()-p.create_time()))
    print('CPU seconds:',p.cpu_times())
    print('Child processes:',[(c.name(),c.cpu_times().user) for c in p.children(recursive=True)])
print(subprocess.check_output(['nvidia-smi','--query-gpu=memory.used,memory.total,utilization.gpu','--format=csv,noheader'],text=True))


In [ ]:
# Runs B and C start sequentially only after a successful baseline.
import threading, traceback
pipeline_status = {'stage':'baseline','pid':train_job.pid}
wide_overrides = ['env.commands.base_velocity.ranges.lin_vel_x=[-2.0,2.0]', 'env.commands.base_velocity.ranges.lin_vel_y=[-2.0,2.0]']
def save_status():
    (artifacts/'status.json').write_text(json.dumps(pipeline_status,indent=2))
def run_wide_experiments():
    global train_job, train_log
    try:
        code = train_job.wait()
        if code != 0:
            raise RuntimeError(f'Baseline failed with exit code {code}; see baseline-console.log')
        for label, iterations in [('wide',300),('wide_long',900)]:
            command = ['uv','run','--no-sync','isaaclab','train','--rl_library','rsl_rl','--task',manifest['task'],'--run_name',label,'--logger','tensorboard','--max_iterations',str(iterations)] + wide_overrides
            (artifacts/f'{label}-command.json').write_text(json.dumps(command,indent=2))
            train_log = artifacts/f'{label}-console.log'
            with train_log.open('w') as log:
                train_job = subprocess.Popen(command,cwd=repo,stdout=log,stderr=subprocess.STDOUT,env={**os.environ,'PYTHONUNBUFFERED':'1'})
                pipeline_status.update(stage=label,pid=train_job.pid); save_status()
                code = train_job.wait()
            if code != 0:
                raise RuntimeError(f'{label} failed with exit code {code}; see {train_log.name}')
        pipeline_status.update(stage='training_complete'); save_status()
    except Exception:
        pipeline_status.update(stage='failed',error=traceback.format_exc()); save_status()
save_status()
experiment_thread = threading.Thread(target=run_wide_experiments,daemon=True)
experiment_thread.start()
print('Sequential experiment controller started:',pipeline_status)


In [ ]:
# Experiment progress. Re-run this cell for an update.
print(pipeline_status)
print(train_log.read_text(errors='replace')[-3500:])
print(subprocess.check_output(['nvidia-smi','--query-gpu=memory.used,memory.total,utilization.gpu','--format=csv,noheader'],text=True))
print('Saved checkpoints:',[str(p.relative_to(repo)) for p in sorted((repo/'logs/rsl_rl/unitree_go2_flat').glob('*/model_*.pt'))][-9:])


In [ ]:
# Analyze and record only after all three training runs finish successfully.
analysis_source = "\"\"\"Export real TensorBoard scalars and compare the three Go2 experiments.\n\nUsage: python analyze_runs.py ISAACLAB_LOG_ROOT OUTPUT_DIRECTORY\nRequires tensorboard, numpy, and matplotlib in the Isaac Lab environment.\n\"\"\"\n\nimport csv\nimport json\nimport sys\nfrom pathlib import Path\n\nimport matplotlib\n\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom tensorboard.backend.event_processing.event_accumulator import EventAccumulator\n\n\ndef analyze(log_root, output):\n    log_root, output = Path(log_root), Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    labels = (\"baseline\", \"wide\", \"wide_long\")\n    metrics = (\"error_vel_xy\", \"success_rate\", \"base_contact\")\n    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))\n    rows, summaries = [], []\n    for label in labels:\n        candidates = sorted(p for p in log_root.iterdir() if p.is_dir() and p.name.endswith(\"_\" + label))\n        if len(candidates) != 1:\n            raise ValueError(f\"Expected exactly one {label} run; found {len(candidates)}\")\n        run = candidates[0]\n        accumulator = EventAccumulator(str(run), size_guidance={\"scalars\": 0})\n        accumulator.Reload()\n        tags = accumulator.Tags()[\"scalars\"]\n        summary = {\"run\": label, \"directory\": run.name, \"window\": \"last 30 logged scalar observations\"}\n        for metric, ax in zip(metrics, axes):\n            matches = [tag for tag in tags if tag.split(\"/\")[-1] == metric]\n            if len(matches) != 1:\n                raise ValueError(f\"Ambiguous or missing {metric} in {run}: {matches}; tags={tags}\")\n            tag = matches[0]\n            events = accumulator.Scalars(tag)\n            if not events:\n                raise ValueError(f\"No scalar observations for {tag} in {run}\")\n            steps = np.array([e.step for e in events])\n            values = np.array([e.value for e in events])\n            if not np.isfinite(values).all():\n                raise ValueError(f\"Non-finite values for {tag} in {run}\")\n            tail = values[-30:]\n            summary[metric] = {\n                \"tag\": tag,\n                \"last_value\": float(values[-1]),\n                \"last_step\": int(steps[-1]),\n                \"tail_mean\": float(tail.mean()),\n                \"tail_std\": float(tail.std()),\n                \"observations\": len(events),\n            }\n            for event in events:\n                rows.append([label, tag, event.step, event.wall_time, event.value])\n            ax.plot(steps, values, alpha=0.18)\n            width = min(15, len(values))\n            smooth = np.convolve(values, np.ones(width) / width, mode=\"valid\")\n            ax.plot(steps[width - 1 :], smooth, label=label)\n        summaries.append(summary)\n    for ax, title, unit in zip(\n        axes,\n        (\"Planar velocity tracking error\", \"Command tracking success\", \"Base-contact termination indicator\"),\n        (\"m/s\", \"Logged fraction\", \"Logged fraction\"),\n    ):\n        ax.set(title=title, xlabel=\"Training iteration\", ylabel=unit)\n        ax.grid(alpha=0.2)\n        ax.legend()\n    fig.suptitle(\"Go2 flat terrain: default vs wider commands (15-observation moving average)\")\n    fig.tight_layout()\n    fig.savefig(output / \"training_comparison.png\", dpi=180)\n    plt.close(fig)\n    with (output / \"scalars.csv\").open(\"w\", newline=\"\", encoding=\"utf-8\") as stream:\n        writer = csv.writer(stream)\n        writer.writerow([\"run\", \"tag\", \"iteration\", \"wall_time\", \"value\"])\n        writer.writerows(rows)\n    (output / \"metrics_summary.json\").write_text(json.dumps(summaries, indent=2), encoding=\"utf-8\")\n    print(json.dumps(summaries, indent=2))\n\n\nif __name__ == \"__main__\":\n    if len(sys.argv) != 3:\n        raise SystemExit(\"Usage: python analyze_runs.py ISAACLAB_LOG_ROOT OUTPUT_DIRECTORY\")\n    analyze(*sys.argv[1:])\n"
(artifacts/'analyze_runs.py').write_text(analysis_source)
import shutil
post_status = {'stage':'waiting_for_training'}
def postprocess():
    global post_log
    try:
        experiment_thread.join()
        if pipeline_status['stage'] != 'training_complete':
            raise RuntimeError('Training did not complete successfully')
        post_status['stage'] = 'analyzing'
        post_log = artifacts/'analysis-console.log'
        with post_log.open('w') as log:
            subprocess.run(['uv','run','--no-sync','python',str(artifacts/'analyze_runs.py'),str(repo/'logs/rsl_rl/unitree_go2_flat'),str(artifacts)],cwd=repo,stdout=log,stderr=subprocess.STDOUT,check=True)
        with (artifacts/'video-setup.log').open('w') as log:
            subprocess.run(['uv','sync','--extra','video'],cwd=repo,stdout=log,stderr=subprocess.STDOUT,check=True)
        for label in ['baseline','wide','wide_long']:
            post_status['stage'] = 'video_' + label
            runs = [p for p in (repo/'logs/rsl_rl/unitree_go2_flat').iterdir() if p.is_dir() and p.name.endswith('_'+label)]
            assert len(runs)==1, runs
            run = runs[0]
            checkpoint = max(run.glob('model_*.pt'),key=lambda p:int(p.stem.split('_')[-1]))
            destination = artifacts/label; destination.mkdir(exist_ok=True)
            shutil.copy2(checkpoint,destination/'checkpoint.pt')
            shutil.copytree(run/'params',destination/'params',dirs_exist_ok=True)
            for event in run.glob('events.out.tfevents.*'):
                shutil.copy2(event,destination/event.name)
            command = ['uv','run','--no-sync','isaaclab','play','--rl_library','rsl_rl','--task',manifest['task'],'--checkpoint',str(checkpoint),'--num_envs','1','--video','newton_gl','--video_length','1000']
            if label != 'baseline': command += wide_overrides
            (destination/'playback-command.json').write_text(json.dumps(command,indent=2))
            post_log = artifacts/f'{label}-video-console.log'
            with post_log.open('w') as log:
                subprocess.run(command,cwd=repo,stdout=log,stderr=subprocess.STDOUT,check=True,env={**os.environ,'PYTHONUNBUFFERED':'1'})
            videos = list(run.rglob('*.mp4'))
            assert videos, 'No video was saved'
            shutil.copy2(max(videos,key=lambda p:p.stat().st_mtime),destination/'policy.mp4')
        post_status['stage'] = 'complete'
    except Exception:
        post_status.update(stage='failed',error=traceback.format_exc())
    (artifacts/'postprocess-status.json').write_text(json.dumps(post_status,indent=2))
post_thread = threading.Thread(target=postprocess,daemon=True)
post_thread.start()
print(post_status)


In [ ]:
# Compact experiment dashboard: safely re-run without starting new training.
import re
print('TRAINING:', pipeline_status)
print('ANALYSIS / VIDEOS:', post_status)
text = train_log.read_text(errors='replace')
iterations = re.findall(r'Learning iteration\s+(\d+/\d+)',text)
print('Latest iteration:',iterations[-1] if iterations else 'initializing')
for metric in ['Metrics/success_rate','Metrics/base_velocity/error_vel_xy','Episode_Termination/base_contact']:
    values = re.findall(re.escape(metric)+r':\s*([-\d.]+)',text)
    print(metric, values[-1] if values else 'pending')
print(text[-600:])
print(subprocess.check_output(['nvidia-smi','--query-gpu=memory.used,utilization.gpu','--format=csv,noheader'],text=True))


In [ ]:
# Serve only this experiment's artifacts through Colab's authenticated proxy.
from google.colab import output
from IPython.display import display, HTML
with (artifacts/'download-server.log').open('w') as log:
    artifact_server = subprocess.Popen([sys.executable,'-m','http.server','6060','--bind','127.0.0.1','--directory',str(artifacts)],stdout=log,stderr=subprocess.STDOUT)
artifact_url = output.eval_js('google.colab.kernel.proxyPort(6060)')
print('Experiment artifact directory:',artifact_url)
display(HTML(f'<a href="{artifact_url}" target="_blank">Open experiment files</a>'))


In [ ]:
# Preserve the original prediction and the exact upstream configuration.
(artifacts/'pretraining-notes.txt').write_text("Senior track: pre-training notes\nRecorded on 20 October 2026, before any training runs.\n\nStatus\nNo training, simulator installation, policy playback, or experiment results have been completed. Hardware compatibility was checked first.\n\nLocal hardware evidence\nnvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv,noheader\nResult: NVIDIA GeForce MX350, 2048 MiB, 537.13, 6.1\n\nThe current official default Isaac Lab environment requires GPU compute capability 7.5 or newer and Windows driver 580.88 or newer. This machine does not meet those requirements. Reducing the environment count does not resolve architecture compatibility. An older custom stack has not been tested.\nSource: https://isaac-sim.github.io/IsaacLab/main/source/isaaclab/installation.html\n\nPrediction before training\nWith the same iteration count and all other settings held constant, expanding both linear command axes from [-1, 1] to [-2, 2] m/s will likely increase planar velocity tracking error and reduce command-tracking success. The task includes harder commands, and a smaller share of samples falls in the original easier range. Base-contact terminations may increase if the learned gait loses stability at higher requested speeds, although that outcome is less certain than the tracking-error change.\n\nTraining for three times as many iterations should improve performance if insufficient optimization or experience is the main limitation. It may not fully close the gap if commands exceed achievable stable speeds under the existing dynamics, action limits, or reward tradeoffs. Improvement alone would not establish a unique cause, and failure to improve would not prove that additional data can never help.\n\nExperiment plan\n1. Record the exact Isaac Lab commit, task ID, solver, seed, default environment count, and default iteration count N.\n2. Train the default configuration for N iterations.\n3. Locate and save the resolved linear velocity command ranges. Change both x and y to [-2, 2], preserving angular command settings and other settings. Check whether a curriculum subsequently changes the ranges.\n4. Train the wide configuration from a fresh initialization for N iterations.\n5. Train the same wide configuration from a fresh initialization for 3N iterations.\n6. Save each final checkpoint, resolved environment and agent configurations, TensorBoard logs, and a video of playback in the requested Newton visualizer.\n7. Compare error_vel_xy, success_rate, and base-contact termination rate. Verify actual tag names and definitions in the installed version; do not substitute reward for success rate. Record missing metrics explicitly.\n8. Report training curves separately from evaluation on a shared command distribution. Metrics from different training command distributions have different difficulty and are not a controlled policy comparison by themselves.\n9. If reducing environment count is necessary, apply the same count to every run and disclose the deviation. Each iteration collects fewer transitions; iteration count alone is not a sample-budget comparison.\n\nResults\nPending real training. No metrics, checkpoints, or videos exist yet.\n\nColab\nA TPU runtime cannot accelerate the required CUDA-based Newton/Warp simulation. A compatible NVIDIA GPU runtime is the route to test. GPU allocation, driver compatibility, package installation, memory capacity, and Newton visualization/video capture must be checked on the actual runtime before claiming success.\n")
shutil.copy2(repo/'uv.lock',artifacts/'upstream_uv.lock')
evidence = artifacts/'upstream_config'; evidence.mkdir(exist_ok=True)
for relative in ['source/isaaclab_tasks/isaaclab_tasks/core/velocity/velocity_env_cfg.py','source/isaaclab_tasks/isaaclab_tasks/core/velocity/config/go2/flat_env_cfg.py','source/isaaclab_tasks/isaaclab_tasks/core/velocity/config/go2/rough_env_cfg.py','source/isaaclab_tasks/isaaclab_tasks/core/velocity/config/go2/agents/rsl_rl_ppo_cfg.py']:
    shutil.copy2(repo/relative,evidence/pathlib.Path(relative).name)
for run in sorted((repo/'logs/rsl_rl/unitree_go2_flat').iterdir()):
    params = run/'params/env.yaml'
    if params.exists():
        contents = params.read_text(); pos = contents.find('lin_vel_x:')
        print(run.name, contents[pos:pos+180])
print('Saved original prediction, upstream lockfile, and configuration sources.')

# Use matching colors for the raw and smoothed curves.
(artifacts/'analyze_runs.py').write_text("\"\"\"Export real TensorBoard scalars and compare the three Go2 experiments.\n\nUsage: python analyze_runs.py ISAACLAB_LOG_ROOT OUTPUT_DIRECTORY\nRequires tensorboard, numpy, and matplotlib in the Isaac Lab environment.\n\"\"\"\n\nimport csv\nimport json\nimport sys\nfrom pathlib import Path\n\nimport matplotlib\n\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom tensorboard.backend.event_processing.event_accumulator import EventAccumulator\n\n\ndef analyze(log_root, output):\n    log_root, output = Path(log_root), Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    labels = (\"baseline\", \"wide\", \"wide_long\")\n    metrics = (\"error_vel_xy\", \"success_rate\", \"base_contact\")\n    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))\n    rows, summaries = [], []\n    for label in labels:\n        candidates = sorted(p for p in log_root.iterdir() if p.is_dir() and p.name.endswith(\"_\" + label))\n        if len(candidates) != 1:\n            raise ValueError(f\"Expected exactly one {label} run; found {len(candidates)}\")\n        run = candidates[0]\n        accumulator = EventAccumulator(str(run), size_guidance={\"scalars\": 0})\n        accumulator.Reload()\n        tags = accumulator.Tags()[\"scalars\"]\n        summary = {\"run\": label, \"directory\": run.name, \"window\": \"last 30 logged scalar observations\"}\n        for metric, ax in zip(metrics, axes):\n            matches = [tag for tag in tags if tag.split(\"/\")[-1] == metric]\n            if len(matches) != 1:\n                raise ValueError(f\"Ambiguous or missing {metric} in {run}: {matches}; tags={tags}\")\n            tag = matches[0]\n            events = accumulator.Scalars(tag)\n            if not events:\n                raise ValueError(f\"No scalar observations for {tag} in {run}\")\n            steps = np.array([e.step for e in events])\n            values = np.array([e.value for e in events])\n            if not np.isfinite(values).all():\n                raise ValueError(f\"Non-finite values for {tag} in {run}\")\n            tail = values[-30:]\n            summary[metric] = {\n                \"tag\": tag,\n                \"last_value\": float(values[-1]),\n                \"last_step\": int(steps[-1]),\n                \"tail_mean\": float(tail.mean()),\n                \"tail_std\": float(tail.std()),\n                \"observations\": len(events),\n            }\n            for event in events:\n                rows.append([label, tag, event.step, event.wall_time, event.value])\n            raw_line, = ax.plot(steps, values, alpha=0.18)\n            width = min(15, len(values))\n            smooth = np.convolve(values, np.ones(width) / width, mode=\"valid\")\n            ax.plot(steps[width - 1 :], smooth, label=label, color=raw_line.get_color())\n        summaries.append(summary)\n    for ax, title, unit in zip(\n        axes,\n        (\"Planar velocity tracking error\", \"Command tracking success\", \"Base-contact termination indicator\"),\n        (\"m/s\", \"Logged fraction\", \"Logged fraction\"),\n    ):\n        ax.set(title=title, xlabel=\"Training iteration\", ylabel=unit)\n        ax.grid(alpha=0.2)\n        ax.legend()\n    fig.suptitle(\"Go2 flat terrain: default vs wider commands (15-observation moving average)\")\n    fig.tight_layout()\n    fig.savefig(output / \"training_comparison.png\", dpi=180)\n    plt.close(fig)\n    with (output / \"scalars.csv\").open(\"w\", newline=\"\", encoding=\"utf-8\") as stream:\n        writer = csv.writer(stream)\n        writer.writerow([\"run\", \"tag\", \"iteration\", \"wall_time\", \"value\"])\n        writer.writerows(rows)\n    (output / \"metrics_summary.json\").write_text(json.dumps(summaries, indent=2), encoding=\"utf-8\")\n    print(json.dumps(summaries, indent=2))\n\n\nif __name__ == \"__main__\":\n    if len(sys.argv) != 3:\n        raise SystemExit(\"Usage: python analyze_runs.py ISAACLAB_LOG_ROOT OUTPUT_DIRECTORY\")\n    analyze(*sys.argv[1:])\n")


In [ ]:
# TensorBoard reads the same original logs used in the report.
from google.colab import output
with (artifacts/'tensorboard-console.log').open('w') as log:
    tensorboard_job = subprocess.Popen(['uv','run','--no-sync','tensorboard','--logdir',str(repo/'logs/rsl_rl/unitree_go2_flat'),'--host','127.0.0.1','--port','6006'],cwd=repo,stdout=log,stderr=subprocess.STDOUT)
print('TensorBoard process:',tensorboard_job.pid)
output.serve_kernel_port_as_iframe(6006,height=650)
